# Passo a Passo Prático: Migração Azure SQL com Unity Catalog
**Engenharia de Dados Aplicada | BRQ Digital Solutions**

## Abordagem Profissional: Unity Catalog e OOP
Para desenvolvedores de software, scripts sequenciais lidando com URIs físicas (`abfss://`) geram alta fricção cognitiva. A melhor prática do mercado é combinar Orientação a Objetos com a abstração lógica do Unity Catalog.

**Vantagens táticas do Unity Catalog:** Governança unificada via SQL (RBAC centralizado), Data Lineage (linhagem de dados) automático, descoberta de dados acelerada no Data Explorer e isolamento da camada física de armazenamento.

## Implementação Prática: Classe AzureSQLMigrator
Crie um arquivo `main.py` com a estrutura modularizada abaixo. A principal melhoria está na padronização da gravação usando o namespace do Unity Catalog.

```python
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp

class AzureSQLMigrator:
    def __init__(self, spark: SparkSession, secret_scope: str, db_server: str, db_name: str):
        self.spark = spark
        self.secret_scope = secret_scope
        self.jdbc_url = f"jdbc:sqlserver://{db_server}:1433;database={db_name}"
        self.dbutils = self._get_dbutils()
        
    def _get_dbutils(self):
        """
        Recupera o contexto do dbutils independentemente de 
        estar rodando em Notebook interativo ou em Job (Workflow).
        """
        try:
            import IPython
            return IPython.get_ipython().user_ns.get("dbutils")
        except Exception:
            from pyspark.dbutils import DBUtils
            return DBUtils(self.spark)

    def extract_data(self, table_name: str, predicate: str = None):
        """Conecta ao banco e realiza a extração. Utiliza predicate para otimizar I/O no relacional."""
        user = self.dbutils.secrets.get(scope=self.secret_scope, key="sql-user")
        password = self.dbutils.secrets.get(scope=self.secret_scope, key="sql-password")
        
        reader = self.spark.read.format("jdbc")\
            .option("url", self.jdbc_url)\
            .option("user", user)\
            .option("password", password)
            
        if predicate:
            reader = reader.option("query", f"SELECT * FROM {table_name} WHERE {predicate}")
        else:
            reader = reader.option("dbtable", table_name)
            
        return reader.load()

    def load_to_bronze_uc(self, df, catalog: str, schema: str, table_name: str):
        """Adiciona metadados e grava no formato Delta abstraído pelo Unity Catalog."""
        full_table_name = f"{catalog}.{schema}.{table_name}"
        
        df_with_metadata = df.withColumn("_ingestion_time", current_timestamp())
        
        # Gravação centralizada gerenciada pelo Unity Catalog
        (df_with_metadata.write
            .format("delta")
            .mode("overwrite")
            .option("mergeSchema", "true")
            .saveAsTable(full_table_name))
        
        # Otimização física através da referência lógica
        self.spark.sql(f"OPTIMIZE {full_table_name}")

# ==========================================
# Entry Point de Execução (Job Execution)
# ==========================================
if __name__ == "__main__":
    print("Inicializando SparkSession...")
    spark = SparkSession.builder.appName("AzureSQL_to_Bronze_UC").getOrCreate()
    
    # Injeção de Dependências
    migrator = AzureSQLMigrator(
        spark=spark,
        secret_scope="kv-adventureworks",
        db_server="<seu-servidor>.database.windows.net",
        db_name="AdventureWorks_LT"
    )
    
    # Parâmetros lógicos (Unity Catalog)
    table_origem = "SalesLT.Customer"
    uc_catalog = "dev_catalog"
    uc_schema = "bronze"
    uc_table = "adventureworks_customers"
    
    print(f"Extraindo dados da tabela {table_origem}...")
    df_clientes = migrator.extract_data(table_name=table_origem)
    
    print(f"Iniciando gravação na tabela {uc_catalog}.{uc_schema}.{uc_table} via Unity Catalog...")
    migrator.load_to_bronze_uc(df_clientes, uc_catalog, uc_schema, uc_table)
    
    print("Pipeline finalizado com sucesso.")